# Initialization

In [8]:
%load_ext autoreload
%autoreload 2
from scipy import stats
import matplotlib.pyplot as plt
import torch
from cleanfid import fid
from metrics.cmmd_pytorch.main import compute_cmmd
from hashlib import md5
import torch.nn.functional as F
from transformers import ViTModel
from PIL import Image

from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel

import os
import shutil
import glob
import torch
import numpy as np
import datetime
from natsort import natsorted
from IPython.display import clear_output
import datetime
import itertools
from pathlib import Path
from collections import defaultdict


from insightface.app import FaceAnalysis
import cv2
import time
from scipy import stats

from tqdm import tqdm as tqmd


device = 'cuda:1'
torch.cuda.set_device(device)
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Current device: 1
Device name: NVIDIA GeForce RTX 3090


In [9]:
data_info = {
    'obama': {'race': 'black', 'gender': 'male', 'unseen': False, 'full_name': 'barrack-obama'},
    'rihanna': {'race': 'black', 'gender': 'female', 'unseen': False, 'full_name': 'rihanna'},
    'edsheeran': {'race': 'white', 'gender': 'male', 'unseen': False, 'full_name': 'ed-sheeran'},
    'mrobbie': {'race': 'white', 'gender': 'female', 'unseen': False, 'full_name': 'margot-robbie'},
    # 'osama': {'race': 'black', 'gender': 'male', 'seen': False},
    # 'honer': {'race': 'white', 'gender': 'male', 'seen': False},
    
    'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
    'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
    'nivola': {'race': 'white', 'gender': 'male','unseen': True},
    'earle': {'race': 'white', 'gender': 'female', 'unseen': True},

    'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
    'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
    'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
    'skyhblack': {'race': 'black', 'gender': 'male', 'unseen': True},

    'sophiewilde':{'race': 'black', 'gender': 'female', 'unseen': True},
    'edebiri':{'race': 'black', 'gender': 'female', 'unseen': True},
    'mmadison': {'race': 'white', 'gender': 'female', 'unseen': True},
    'nicoparker': {'race': 'white', 'gender': 'female', 'unseen': True},

    'chemsworth': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-hemsworth'},  # Chris Hemsworth
    'cevans': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-evans'},      # Chris Evans
    # 'adriver': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'adam-driver'},     # Adam Driver
    # 'agarfield': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'andrew-garfield'},   # Andrew Garfield
    
    'aadam': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-adam'},       # Anne Adam
    'ahathaway': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-hathaway'}, # Anne Hathaway
    # 'ajolie': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'angelina-jolie'},    # Angelina Jolie
    # 'amber': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'amber-heard'},     # Likely Amber Heard
    
    'mcarey': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'mariah-carey'},    # Mariah Carey (black heritage)
    'octavia': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'octavia-spencer'},   # Octavia Spencer
    # 'oprah': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'oprah-winfrey'},     # Oprah Winfrey
    
    'morganf': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'morgan-freeman'},     # Morgan Freeman
    'drake': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'drake'},       # Drake (mixed but usually listed as black)
    # 'idris': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'idris-elba'},       # Idris Elba
    
}

learn_concepts =  []; rl_concepts = []
for concept, info in data_info.items():
    if info['unseen']:
        learn_concepts.append(concept)
    else:
        rl_concepts.append(concept)

small_rl_concepts = ["obama","rihanna","edsheeran","mrobbie","chemsworth","aadam","mcarey","morganf"] # "cevans-starkey", "aadam-apierre", "ahathaway-skyhblack",
small_learn_concepts = ["asante", "reese", "nivola", "earle","leowoodal",'mmadison','sophiewilde',"apierre"] #,"starkey","apierre","skyhblack","sophiewilde","edebiri","mmadison","nicoparker"] # "cevans-starkey", "aadam-apierre", "ahathaway-skyhblack",
small_rlx_concepts = ["obama-asante", "rihanna-reese", "edsheeran-nivola", "mrobbie-earle","chemsworth-leowoodal","aadam-mmadison","mcarey-sophiewilde","morganf-apierre"] # "cevans-starkey", "aadam-apierre", "ahathaway-skyhblack",


In [10]:
import numpy as np
import matplotlib.pyplot as plt

def plot_mean_confidence(
    data: dict,
    steps,
    confidence: float = 90,
    xlabel: str = "Training step",
    ylabel: str = "Score",
    title: str | None = None,
    alpha: float = 0.25,
    figsize: tuple[int, int] = (7, 4.5),
):
    """
    Plot mean ± CI curves for multiple experiments.

    Parameters
    ----------
    data : dict[str, list[list[float]]]
        Mapping {experiment_name: runs}.  Each value must be a list (or
        2‑D array) of shape (n_runs, n_steps).
    steps : 1‑D sequence
        X‑axis labels (must match length of a single run).
    confidence : float
        Confidence level in percent (common choices: 99, 97.5, 95, 90).
    xlabel, ylabel, title : str
        Axis labels and figure title.
    alpha : float
        Transparency of the CI band (0–1).
    figsize : tuple[int, int]
        Figure size in inches.

    Returns
    -------
    (fig, ax) : matplotlib Figure and Axes.
    """
    # critical z for usual two‑sided CIs
    _z_table = {90: 1.645, 95: 1.960, 97.5: 2.241, 99: 2.576}
    if confidence in _z_table:
        z = _z_table[confidence]
    else:                                        # fallback to scipy if available
        try:
            from scipy.stats import norm
            z = norm.ppf(0.5 + confidence / 100 / 2)
        except ImportError as e:
            raise ValueError(
                f"Unsupported confidence={confidence}. "
                f"Install SciPy or use one of {_z_table.keys()}."
            ) from e

    steps = np.asarray(steps)
    fig, ax = plt.subplots(figsize=figsize)
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for i, (name, runs) in enumerate(data.items()):
        runs = np.asarray(runs, dtype=float)
        if runs.ndim != 2:
            raise ValueError(f"{name}: expected shape (n_runs, n_steps), got {runs.shape}")
        if runs.shape[1] != len(steps):
            raise ValueError(f"{name}: len(steps)={len(steps)} ≠ n_steps={runs.shape[1]}")

        mean = runs.mean(0)
        sem  = runs.std(0, ddof=1) / np.sqrt(runs.shape[0])
        ci   = z * sem

        color = color_cycle[i % len(color_cycle)]
        ax.plot(steps, mean, label=name, color=color)
        ax.fill_between(steps, mean - ci, mean + ci, color=color, alpha=alpha)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title or f"Metric vs. step (±{confidence}% CI)")
    ax.grid(True)
    ax.legend()
    # plt.tight_layout()
    
    
    # ── NEW: note about the shaded band ──────────────────────────
    n_seeds = len(data[list(data.keys())[0]])
    ci_note = f"Shaded band = ±{confidence}% CI (n={len(data[list(data.keys())[0]])})"
    ax.text(0.02, 0.02, ci_note,
            transform=ax.transAxes,          # axes coords (0–1)
            ha="left", va="bottom",
            fontsize=8, color="gray")
    # ─────────────────────────────────────────────────────────────

    plt.tight_layout()
    
    return fig, ax




def list_images_in_folder(folder_path, extensions={'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}):
    """
    Returns a list of image file paths from the given folder.
    
    Args:
        folder_path (str): Path to the folder.
        extensions (set): Set of allowed image file extensions (case-insensitive).

    Returns:
        List[str]: List of full paths to image files.
    """
    image_files = []
    for fname in os.listdir(folder_path):
        full_path = os.path.join(folder_path, fname)
        if os.path.isfile(full_path) and os.path.splitext(fname)[1].lower() in extensions:
            image_files.append(full_path)
    return image_files

def get_file_creation_date(filepath):
    timestamp = os.path.getctime(filepath)
    dt = datetime.datetime.fromtimestamp(timestamp)
    return dt.strftime("%d-%m-%y_%H:%M")

def calc_cdist_part(features_1, features_2, batch_size=10000):
    dists = []
    for feat2_batch in features_2.split(batch_size):
        dists.append(torch.cdist(features_1, feat2_batch).cpu())
    return torch.cat(dists, dim=1)


def calculate_precision_recall_part(features_1, features_2, neighborhood=3, batch_size=10000):
    # Precision
    dist_nn_1 = []
    for feat_1_batch in features_1.split(batch_size):
        dist_nn_1.append(calc_cdist_part(feat_1_batch, features_1, batch_size).kthvalue(neighborhood + 1).values)
    dist_nn_1 = torch.cat(dist_nn_1)
    precision = []
    for feat_2_batch in features_2.split(batch_size):
        dist_2_1_batch = calc_cdist_part(feat_2_batch, features_1, batch_size)
        precision.append((dist_2_1_batch <= dist_nn_1).any(dim=1).float())
    precision = torch.cat(precision).mean().item()
    # Recall
    dist_nn_2 = []
    for feat_2_batch in features_2.split(batch_size):
        dist_nn_2.append(calc_cdist_part(feat_2_batch, features_2, batch_size).kthvalue(neighborhood + 1).values)
    dist_nn_2 = torch.cat(dist_nn_2)
    recall = []
    for feat_1_batch in features_1.split(batch_size):
        dist_1_2_batch = calc_cdist_part(feat_1_batch, features_2, batch_size)
        recall.append((dist_1_2_batch <= dist_nn_2).any(dim=1).float())
    recall = torch.cat(recall).mean().item()
    return precision, recall


def get_features(base_dir, use_precompute_features_if_exist=False,max_count=-1):
    # TODO: right now, expect the features to be already exist only
    image_extensions = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(base_dir, ext)))

    if not image_files:
        raise FileNotFoundError(f"No image files found in {base_dir}")

    first_image_path = natsorted(image_files)[0]
    date_str = get_file_creation_date(first_image_path)
        
    if use_precompute_features_if_exist:
        feature_dir = os.path.join(base_dir, "precomputed_features", "clip-l-14")
        # feature_dir = os.path.join(base_dir, "precomputed_features", "clip-b-32")
        feature_filename = f"{date_str}_n{max_count}.npy"
        feature_path = os.path.join(feature_dir, feature_filename)
        # feature_path = feature_path.replace("+", "--")  # Replace ':' with '-' for filename compatibility

        if os.path.exists(feature_path):
            print(f"Loading precomputed features from {feature_path}")
            feature = np.load(feature_path).astype("float32")
            feature = torch.tensor(feature)
            return feature
        else: print(f'{feature_path} does not exist')
            

def compute_pr(ref_path, eval_path, k=3, use_precompute_features_if_exist=False, feature=None, max_count=None):
    ref_features = get_features(ref_path, use_precompute_features_if_exist=use_precompute_features_if_exist, max_count=max_count)
    eval_features = get_features(eval_path, use_precompute_features_if_exist=use_precompute_features_if_exist, max_count=max_count)
    precision, recall = calculate_precision_recall_part(ref_features, eval_features, neighborhood=k)

    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    return precision, recall, f1

def list_images_in_folder(folder_path, extensions={'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}):
    """
    Returns a list of image file paths from the given folder.
    
    Args:
        folder_path (str): Path to the folder.
        extensions (set): Set of allowed image file extensions (case-insensitive).

    Returns:
        List[str]: List of full paths to image files.
    """
    image_files = []
    for fname in os.listdir(folder_path):
        full_path = os.path.join(folder_path, fname)
        if os.path.isfile(full_path) and os.path.splitext(fname)[1].lower() in extensions:
            image_files.append(full_path)
    return image_files



def plot_multiple_score_curves(steps, all_scores, labels=None, title="Score vs Training Steps", save_path=None, method='KID'):
    plt.figure(figsize=(8, 5))
    for idx, scores in enumerate(all_scores):
        label = labels[idx] if labels else f"Exp {idx+1}"
        plt.plot(steps, scores, marker='o', linestyle='-', label=label)
    plt.xlabel("Training Step")
    plt.ylabel(f"{method.upper()} Score")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# obsolete
def create_limited_folder(original_path, limit, dummy_path):
    os.makedirs(dummy_path, exist_ok=True)
    image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp')
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(original_path, ext)))
    image_files = sorted(image_files)[:limit]
    for img_file in image_files:
        if os.path.isfile(img_file):
            shutil.copy(img_file, dummy_path)

def flatten_path(path):
    return os.path.normpath(path).replace(os.sep, '__')

def get_first_image_creation_date(folder):
    image_files = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp'):
        image_files.extend(glob.glob(os.path.join(folder, ext)))
    if not image_files:
        raise FileNotFoundError(f"No images found in {folder}")
    first_img = natsorted(image_files)[0]
    ts = os.path.getctime(first_img)
    return datetime.datetime.fromtimestamp(ts).strftime("%d-%m-%y_%H:%M")


def get_safe_filename(original_str):
    hash_id = md5(original_str.encode()).hexdigest()
    return hash_id

def build_score_cache_path(method, eval_path, ref_path, n_max_gen_img,apply_hash=True, other_params={}, compute_option=None):
    f_eval = flatten_path(eval_path)
    f_ref  = flatten_path(ref_path)
    t_eval = get_first_image_creation_date(eval_path)
    t_ref  = get_first_image_creation_date(ref_path)
    filename = f"{f_eval}:{t_eval}--{f_ref}:{t_ref}_n{n_max_gen_img}"
    
    if method == 'pr' and "neighborhood" in other_params and other_params["neighborhood"] != 3:
        filename += f"_k{other_params['neighborhood']}"
        
        
        
    if method == 'arcface' or method == 'dinov2':
        if compute_option is not None:
            filename += f"_option-{compute_option}"
        if "treat_undetected_as_negative" in other_params and other_params["treat_undetected_as_negative"]:
            filename += "_udneg"
    
    if apply_hash:
        # print('applying hash to filename')
        old_filename = filename
        filename = get_safe_filename(filename)
        # print(f"Old filename: {old_filename} -> New filename: {filename}")
    filename = f"{filename}.npy"
    return os.path.join("data_root", "cache", "precomputed_scores", method, filename)

def compute_distribution_score_multiexp(
    gen_img_paths,
    ref_img_path,
    steps,
    labels=None,
    device='cuda',
    n_max_gen_img=None,
    method='kid',
    use_precompute_features_if_exist=False,
    use_precompute_score_if_exist=False,
    clear_notebook_output=True,
    compute_option='pairwise',
    other_params={},
):
    if labels and len(labels) != len(gen_img_paths):
        raise ValueError("The number of labels must match the number of generated image paths.")

    all_scores = []

    for gen_path in gen_img_paths:
        scores = []
        for step in steps:
            path_for_eval = gen_path.format(step) if "{}" in gen_path else gen_path
            if n_max_gen_img: assert len(list_images_in_folder(path_for_eval)) >= n_max_gen_img
            
            cache_path = build_score_cache_path(method, path_for_eval, ref_img_path, n_max_gen_img,other_params=other_params, compute_option=compute_option)
            # cache_path = cache_path.replace("+", "--")  # Replace ':' with '-' for filename compatibility
            if use_precompute_score_if_exist:
                if os.path.exists(cache_path):
                    if method == 'pr':
                        score = np.load(cache_path)
                    else:
                        score = np.load(cache_path).item()
                    print(f"[CACHED] {method.upper()} from {cache_path}")
                else:
                    score = _compute(method, ref_img_path, path_for_eval, device, n_max_gen_img, use_precompute_features_if_exist, other_params=other_params, compute_option=compute_option)
                    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
                    np.save(cache_path, score)
                    print(f"[SAVED] {method.upper()} to {cache_path}")
            else:
                score = _compute(method, ref_img_path, path_for_eval, device, n_max_gen_img, use_precompute_features_if_exist, other_params=other_params, compute_option=compute_option)
                os.makedirs(os.path.dirname(cache_path), exist_ok=True)
                np.save(cache_path, score)
                print(f"[SAVED] {method.upper()} to {cache_path}")
                
            if "{}" not in gen_path:
                print(f'{gen_path} | {method.upper()}: {score}')
                scores = [score] * len(steps)
                break
            else:
                print(f'{gen_path} | Step {step} - {method.upper()}: {score}')
                scores.append(score)

        all_scores.append(scores)

    if clear_notebook_output: clear_output(wait=True)
    if method == 'pr':
        Precisions = [[p for p, _, _ in sublist] for sublist in all_scores]
        Recalls = [[r for _, r, _ in sublist] for sublist in all_scores]
        F1s = [[f for _, _, f in sublist] for sublist in all_scores]
        plot_multiple_score_curves(steps, Precisions, labels, method='precision')
        plot_multiple_score_curves(steps, Recalls, labels, method='recall')
        plot_multiple_score_curves(steps, F1s, labels, method='f1')
    else:
        
        plot_multiple_score_curves(steps, all_scores, labels, method=method)
    return all_scores

def _compute(method, ref_path, eval_path, device, n_max_gen_img, use_precompute_features_if_exist,other_params={}, compute_option='pairwise'):
    if method == 'arcface':
        average_similarity, similarities, stats = arcface_pipeline.compute_arcface_similarity(eval_path, ref_path, compute_option=compute_option, other_params=other_params)
        # print(f" arcface_pipeline.compute_arcface_similarity time: {time2:.4f}s")
        return average_similarity
    elif method == 'dinov2':
        average_similarity, similarities, stats = dinoface_pipeline.compute_dino_face_similarity(eval_path, ref_path, other_params=other_params)
        return average_similarity
    elif method == 'kid':
        return fid.compute_kid(eval_path, ref_path, n_max_gen_img=n_max_gen_img, device=device, use_dataparallel=False)
    elif method == 'cmmd':
        return compute_cmmd(
            ref_path,
            eval_path,
            batch_size=10,
            max_count=-1 if n_max_gen_img is None else n_max_gen_img,
            use_precompute_features_if_exist=use_precompute_features_if_exist
        ).item()
    elif method == 'pr':
        print(f"k: {other_params.get('neighborhood', 3)}")
        return compute_pr(ref_path, eval_path, k=other_params.get('neighborhood', 3), max_count=n_max_gen_img,use_precompute_features_if_exist=use_precompute_features_if_exist)
    else:
        raise ValueError(f"Unsupported method: {method}")
    

In [11]:

        
import math
from statistics import mean, stdev
from typing import Sequence, Tuple


def area_under_curve(scores, steps, *, normalise: bool = False, use_torch: bool = False):
    """
    Compute the area under a score‑versus‑training‑step curve.

    Parameters
    ----------
    scores : Sequence[float] | np.ndarray | torch.Tensor
        Metric values measured along training.
    steps  : Sequence[float] | np.ndarray | torch.Tensor
        Corresponding training‑step (or epoch) indices.
    normalise : bool, optional (default False)
        If True, divide by (steps[-1] - steps[0]) so the result is in [0, 1].
    use_torch : bool, optional (default False)
        If True, do the computation with torch.*; otherwise NumPy is used.

    Returns
    -------
    float
        Area under the curve (normalised if `normalise=True`).

    Notes
    -----
    • Uses the trapezoidal rule.  
    • Requires `scores` and `steps` to have the same length
      and `steps` to be strictly increasing.
    """
    if len(scores) != len(steps):
        raise ValueError("`scores` and `steps` must have the same length.")

    if use_torch:
        scores_t = torch.as_tensor(scores, dtype=torch.float64)
        steps_t  = torch.as_tensor(steps,  dtype=torch.float64)
        auc = torch.trapz(scores_t, steps_t).item()
    else:
        scores_n = np.asarray(scores, dtype=np.float64)
        steps_n  = np.asarray(steps,  dtype=np.float64)
        auc = np.trapz(scores_n, steps_n)

    if normalise:
        span = float(steps[-1] - steps[0])
        if span == 0:
            raise ZeroDivisionError("Cannot normalise when all steps are identical.")
        auc /= span
    return auc

    
def compute_poa_trapz(r, p, training_steps):
    #precision_outperformance_area_trapz
    """
    Compute POA using the trapezoidal rule.
    
    Args:
        r (list or np.array): Precision values of method u(s)
        p (list or np.array): Precision values of baseline p(s)
        training_steps (list or np.array): List of training steps s
    
    Returns:
        float: POA value
    """
    r, p, training_steps = np.array(r), np.array(p), np.array(training_steps)
    f = np.maximum(r - p, 0)  # f(s) = max(f(s) - p(s), 0)
    
    # print(r)
    # print(p)
    # print(f)
    
    return np.trapz(f, training_steps)

    # f"data_root/generated/model/c.l4.kv_reeseU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}",
    # f"data_root/generated/model/rl4.reV.reeseU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}",
    
    # f"data_root/generated/model/rl4.reV.reeseU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}",
    # f"data_root/generated/model/r
    # 
    # l4.reV.reeseU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.gout.person.s50_c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}",
    
def load_precision_cache(gen_img_template_path, ref_img_path, n_max_gen_img=50, training_steps=None):
    scores = []
    for step in training_steps:
        gen_img_path = gen_img_template_path.format(step)
        # print(gen_img_path,ref_img_path)
        cache_path = build_score_cache_path('pr', gen_img_path, ref_img_path, n_max_gen_img)
        score = np.load(cache_path)
        pr,rc,f1 = score
        scores += [pr]
    return scores


def load_arcface_cache(gen_img_template_path, ref_img_path, n_max_gen_img=5, training_steps=None, compute_option='avg_ref', other_params={}):
    scores = []
    for step in training_steps:
        gen_img_path = gen_img_template_path.format(step)
        cache_path = build_score_cache_path('arcface', gen_img_path, ref_img_path, n_max_gen_img, compute_option=compute_option, other_params=other_params)
        score = np.load(cache_path)
        scores += [score]
    return scores


def load_dino_cache(gen_img_template_path, ref_img_path, n_max_gen_img=5, training_steps=None, compute_option='pairwise', other_params={}):
    scores = []
    for step in training_steps:
        gen_img_path = gen_img_template_path.format(step)
        # print(gen_img_path,ref_img_path)
        cache_path = build_score_cache_path('dinov2', gen_img_path, ref_img_path, n_max_gen_img, compute_option=compute_option, other_params=other_params)
        score = np.load(cache_path)
        scores += [score]
    return scores


def compute_poa_template(base_concept='chiquita',all_concepts=['chiquita','reese','jooli','gout','honer'],training_steps=list(range(0,1001,100))):
    lrs_lora = ["5e-4","1e-4","5e-5"]
    base_cfg = 7.5
    
    for lr in lrs_lora:
        ref_img_path = f"data_root/data/real_data/{base_concept}/{base_concept}-50"
        n_max_gen_img = 50
        
        # pretrained_path = f"data_root/generated/model/c.l4.kv_{base_concept}U3-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
        # relearn_path = f"data_root/generated/model/rl4.reV.{base_concept}U3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.{base_concept}.person.s50_c.l4.kv_{base_concept}50-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
        
        # pretrained_precisions = load_precision_cache(pretrained_path, ref_img_path, n_max_gen_img=50, training_steps=training_steps)
        # relearn_precisions = load_precision_cache(relearn_path, ref_img_path, n_max_gen_img=50, training_steps=training_steps)
        
        # poa_relearn = compute_poa_trapz(relearn_precisions, pretrained_precisions, training_steps)
        
        # print(f"POA for unlearn {base_concept} -> learn {base_concept}  lr{lr} relearn: {poa_relearn:.4f}")
        
        
        seed_pretrained_precisions = []; seed_relearn_precisions = []
        for seed in [0,1,2]:
            if seed !=0:
                pretrained_path = f"data_root/generated/model/c.l4.kv_{base_concept}U3-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4.r{seed}/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
                relearn_path = f"data_root/generated/model/rl4.reV.{base_concept}U3.r{seed}_ul1.prg1e-4d8e+3.lr1e-4.n8.G.{base_concept}.person.s50_c.l4.kv_{base_concept}50-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
            else:
                pretrained_path = f"data_root/generated/model/c.l4.kv_{base_concept}U3-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
                relearn_path = f"data_root/generated/model/rl4.reV.{base_concept}U3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.{base_concept}.person.s50_c.l4.kv_{base_concept}50-V_pr0.50_lr{lr}.ti5e-2_f0.5_b1g4.s3000/checkpoint-{{}}/A photo of a v1/{base_cfg:.2f}"
                
            seed_pretrained_precisions += [load_precision_cache(pretrained_path, ref_img_path, n_max_gen_img=50, training_steps=training_steps)]
            seed_relearn_precisions += [load_precision_cache(relearn_path, ref_img_path, n_max_gen_img=50, training_steps=training_steps)]
            
        
        avg_pretrained_precisions = np.mean(seed_pretrained_precisions, axis=0)
        avg_relearn_precisions = np.mean(seed_relearn_precisions, axis=0)
        
        
        
        
        poa_relearn = compute_poa_trapz(avg_relearn_precisions, avg_pretrained_precisions, training_steps)
            
        print(f"POA for unlearn {base_concept} -> learn {base_concept}  lr{lr} relearn: {poa_relearn:.4f}")
        
        
        
        
        # # graph
        Precisions = [avg_pretrained_precisions,avg_relearn_precisions]
        labels = [f'Pretrained {base_concept}', f'Relearn {base_concept}']
        
        plot_multiple_score_curves(training_steps, Precisions, labels, method='precision', title=f"POA for unlearn {base_concept} -> learn {base_concept}  lr{lr} relearn: {poa_relearn:.4f}")
        

        


def mean_se_ci(
    scores: Sequence[float], confidence: float = 0.95
) -> Tuple[float, float, Tuple[float, float]]:
    """
    Parameters
    ----------
    scores : Sequence[float]
        1‑D collection of numbers (e.g. [0.1, 0.15, 0.2, 0.2]).
    confidence : float, optional
        Confidence level for the interval (default 0.95 ⇒ 95 % CI).

    Returns
    -------
    mean_ : float
        Sample mean.
    se : float
        Standard error of the mean.
    ci : (float, float)
        Two‑sided confidence interval (lower, upper) at the requested level.
    """
    n = len(scores)
    if n < 2:
        raise ValueError("At least two observations are required.")

    # Sample mean and standard error
    mean_ = mean(scores)
    se = stdev(scores) / math.sqrt(n)      # uses unbiased stdev (ddof=1)

    # Two‑sided t‑interval (Student‑t accounts for small‑sample uncertainty)
    alpha = 1.0 - confidence
    from math import erf, sqrt

    # quick inline t‑quantile via inverse‑error‑function approximation
    def t_ppf(p: float, df: int) -> float:
        """Approximate Student‑t quantile using an ERF‑based formula."""
        # Abramowitz & Stegun 26.2.23 for df≥1 (good enough for CI)
        a = 1.0 / (4 * df)
        return sqrt(df) * math.tan(math.pi * (p - 0.5)) * (
            1 + a * ( (math.tan(math.pi * (p - 0.5))) ** 2)
        )

    t_crit = t_ppf(1 - alpha / 2, n - 1)
    margin = t_crit * se
    ci = (mean_ - margin, mean_ + margin)

    return mean_, se, ci



In [12]:
class ArcFacePipeline:
    def __init__(self, arcface_app):
        """
        Initialize the ArcFace pipeline
        
        Args:
            arcface_app: Pre-initialized ArcFace application for face detection and recognition
        """
        self.arcface_app = arcface_app
    
    def detect_and_crop_face(self, image_path, face_index=0):
        """
        Detect face in image and return cropped face
        
        Args:
            image_path (str): Path to input image
            face_index (int): Index of face to use if multiple faces detected
            
        Returns:
            PIL.Image: Cropped face image
            dict: Face detection info
        """
        # Read image using cv2 (BGR format) for face detection
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise ValueError(f"Could not read image: {image_path}")
        
        # Detect faces using BGR image
        faces = self.arcface_app.get(img_bgr)
        
        if len(faces) < 1:
            raise ValueError("No faces detected in the image")
        
        if len(faces) > 1:
            print(f"Warning: {len(faces)} faces detected. Using face at index {face_index}")
        
        if face_index >= len(faces):
            raise ValueError(f"Face index {face_index} out of range. Only {len(faces)} faces detected")
        
        # Get selected face bbox
        face = faces[face_index]
        bbox = face.bbox.astype(int)
        x1, y1, x2, y2 = bbox
        
        # Crop face from BGR image
        face_crop_bgr = img_bgr[y1:y2, x1:x2]
        
        # Convert BGR to RGB for PIL processing
        face_crop_rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
        face_pil = Image.fromarray(face_crop_rgb)
        
        face_info = {
            'bbox': bbox,
            'confidence': face.det_score,
            'face_index': face_index,
            'total_faces': len(faces),
            'face_crop_rgb': face_crop_rgb,
            'face_crop_pil': face_pil,
            'embedding': face.embedding
        }
        
        return face_pil, face_info

    def batch_detect_and_crop_face(self, image_paths, face_index=0, batch_size=32):
        """
        Optimized batch processing for face detection and cropping.
        Since InsightFace doesn't have native batch processing, this implements
        optimizations like pre-loading images and reducing I/O overhead.
        
        Args:
            image_paths (list): List of image file paths
            face_index (int): Index of face to use if multiple faces detected
            batch_size (int): Number of images to process in each batch (for memory management)
            
        Returns:
            tuple: (successful_results, failed_results)
                - successful_results: Dict mapping image_path -> (face_pil, face_info)
                - failed_results: Dict mapping image_path -> error_message
        """
        successful_results = {}
        failed_results = {}
        
        print(f"Processing {len(image_paths)} images in batches of {batch_size} (optimized I/O)")
        
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]
            
            print(f"Processing batch {i//batch_size + 1}/{(len(image_paths) + batch_size - 1)//batch_size} "
                  f"({len(batch_paths)} images)")
            
            # Pre-load all images in the batch to reduce I/O overhead
            batch_data = []
            for img_path in batch_paths:
                try:
                    img_bgr = cv2.imread(img_path)
                    if img_bgr is not None:
                        batch_data.append((img_path, img_bgr))
                    else:
                        failed_results[img_path] = f"Could not read image: {img_path}"
                except Exception as e:
                    failed_results[img_path] = f"Error loading image: {str(e)}"
            
            if not batch_data:
                continue
            
            # Process each image in the pre-loaded batch
            # Note: InsightFace doesn't support true batch inference with app.get()
            # But we can optimize by reducing I/O and processing in chunks
            for img_path, img_bgr in batch_data:
                try:
                    # Use the standard InsightFace get() method
                    faces = self.arcface_app.get(img_bgr)
                    
                    if len(faces) < 1:
                        failed_results[img_path] = "No faces detected in the image"
                        continue
                    
                    if len(faces) > 1:
                        print(f"Warning: {len(faces)} faces detected in {img_path}. Using face at index {face_index}")
                    
                    if face_index >= len(faces):
                        failed_results[img_path] = f"Face index {face_index} out of range. Only {len(faces)} faces detected"
                        continue
                    
                    # Get selected face
                    face = faces[face_index]
                    bbox = face.bbox.astype(int)
                    x1, y1, x2, y2 = bbox
                    
                    # Crop face from BGR image
                    face_crop_bgr = img_bgr[y1:y2, x1:x2]
                    
                    # Convert BGR to RGB for PIL processing
                    face_crop_rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
                    face_pil = Image.fromarray(face_crop_rgb)
                    
                    # display(face_pil)
                    
                    
                    face_info = {
                        'bbox': bbox,
                        'confidence': face.det_score,
                        'face_index': face_index,
                        'total_faces': len(faces),
                        'face_crop_rgb': face_crop_rgb,
                        'face_crop_pil': face_pil,
                        'embedding': face.embedding,
                        'image_path': img_path
                    }
                    # print(face.embedding)
                    successful_results[img_path] = (face_pil, face_info)
                    
                except Exception as e:
                    failed_results[img_path] = f"Face processing error: {str(e)}"
            
            # Optional: Force garbage collection after each batch to manage memory
            import gc
            gc.collect()
        
        print(f"Batch processing complete: {len(successful_results)} successful, {len(failed_results)} failed")
        return successful_results, failed_results
    
    def get_face_embedding(self, image_path, face_index=0):
        """
        Extract face embedding from an image
        
        Args:
            image_path (str): Path to input image
            face_index (int): Index of face to use if multiple faces detected
            
        Returns:
            np.ndarray: Face embedding
        """
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Could not read image: {image_path}")

        faces = self.arcface_app.get(img)

        if len(faces) < 1:
            raise ValueError("No faces detected in the image")
        if len(faces) > 1:
            print(f"Warning: Multiple faces detected. Using face at index {face_index}")
        
        if face_index >= len(faces):
            raise ValueError(f"Face index {face_index} out of range. Only {len(faces)} faces detected")
            
        return faces[face_index].embedding
    
    def compare_faces(self, emb1, emb2, threshold=0.65):
        """
        Compare two embeddings using cosine similarity
        
        Args:
            emb1: First face embedding
            emb2: Second face embedding
            threshold: Decision threshold (default 0.65 is common for ArcFace)
            
        Returns:
            float: Cosine similarity value (0-1)
        """
        similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
        return similarity
    
    def compute_face_similarity(self, image_path1, image_path2, face_index1=0, face_index2=0):
        """
        Compute similarity between faces in two images
        
        Args:
            image_path1 (str): Path to first image
            image_path2 (str): Path to second image
            face_index1 (int): Face index to use from first image
            face_index2 (int): Face index to use from second image
            
        Returns:
            dict: Results including similarity score and face info
        """
        # Detect and crop faces
        face1, face1_info = self.detect_and_crop_face(image_path1, face_index1)
        face2, face2_info = self.detect_and_crop_face(image_path2, face_index2)
        
        # Get embeddings from face info (already computed during detection)
        emb1 = face1_info['embedding']
        emb2 = face2_info['embedding']
        
        # Normalize embeddings
        emb1_norm = emb1 / np.linalg.norm(emb1)
        emb2_norm = emb2 / np.linalg.norm(emb2)
        
        # Compute cosine similarity
        similarity = self.compare_faces(emb1_norm, emb2_norm)
        
        results = {
            'similarity': similarity,
            'face1_info': face1_info,
            'face2_info': face2_info,
            'face1_crop': face1,
            'face2_crop': face2,
            'embedding1': emb1,
            'embedding2': emb2
        }
        
        return results
    
    def get_image_files(self, directory):
        """
        Get all image files from a directory
        
        Args:
            directory (str): Path to directory
            
        Returns:
            list: List of image file paths
        """
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
        image_files = []
        
        for file_path in Path(directory).rglob('*'):
            if file_path.is_file() and file_path.suffix.lower() in image_extensions:
                image_files.append(str(file_path))
        
        return image_files
    def compute_arcface_similarity(self, generated_path, ref_path, compute_option='avg_ref', batch_size=32, other_params={}):
        """
        Compute ArcFace similarity scores between generated and reference images using batch processing
        
        Args:
            generated_path (str): Directory containing generated images
            ref_path (str): Directory containing reference images
            compute_option (str): Method to compute similarity:
                'avg_ref' - compares against average reference embedding (default)
                'pairwise' - compares against each reference image individually
            batch_size (int): Batch size for processing images
        
        Returns:
            tuple: (average_similarity, individual_scores, stats)
                - average_similarity: Average cosine similarity score
                - individual_scores: List of similarity scores (method depends on compute_option)
                - stats: Dictionary containing additional statistics
        """
        # Get all image files from both directories
        generated_images = natsorted(self.get_image_files(generated_path))
        ref_images = self.get_image_files(ref_path)
        
        if not generated_images:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_images:
            raise ValueError(f"No images found in reference directory: {ref_path}")
        
        print(f"Found {len(generated_images)} generated images and {len(ref_images)} reference images")
        
        # Batch process reference images
        print("Processing reference images...")
        ref_results, ref_failed = self.batch_detect_and_crop_face(ref_images, batch_size=batch_size)
        
        if not ref_results:
            raise ValueError("No valid face embeddings extracted from reference images")
        
        # Extract reference embeddings
        ref_embeddings = []
        for img_path, (face_pil, face_info) in ref_results.items():
            ref_embeddings.append(face_info['embedding'])
        
        # Normalize all reference embeddings
        ref_embeddings = np.array(ref_embeddings)
        ref_embeddings_norm = ref_embeddings
        # ref_embeddings_norm = np.array([emb / np.linalg.norm(emb) for emb in ref_embeddings])
        
        # Batch process generated images
        print("Processing generated images...")
        gen_results, gen_failed = self.batch_detect_and_crop_face(generated_images, batch_size=batch_size)
        
        if not gen_results:
            raise ValueError("No valid similarities computed from generated images")
        
        # Compute similarities for each generated image
        similarities = []
        pairwise_similarities = []  # Only used for 'pairwise' mode
        
        print("Computing similarities...")
        for i, (img_path, (face_pil, face_info)) in enumerate(gen_results.items()):
            generated_embedding = face_info['embedding']
            generated_embedding_norm = generated_embedding / np.linalg.norm(generated_embedding)
            
            if compute_option == 'avg_ref':
                # Compare to average reference embedding
                avg_ref_embedding = np.mean(ref_embeddings_norm, axis=0)
                avg_ref_embedding = avg_ref_embedding / np.linalg.norm(avg_ref_embedding)
                similarity = self.compare_faces(generated_embedding_norm, avg_ref_embedding)
                similarities.append(similarity)
                
            elif compute_option == 'pairwise':
                # Compare to each reference individually using vectorized operations
                pairwise_sims = []
                for ref_emb in ref_embeddings_norm:
                    sim = self.compare_faces(generated_embedding_norm, ref_emb)
                    pairwise_sims.append(sim)
                
                pairwise_similarities.append(pairwise_sims)
                similarities.append(np.mean(pairwise_sims))  # Average of all pairwise similarities
                
            else:
                raise ValueError(f"Invalid compute_option: {compute_option}. Must be 'avg_ref' or 'pairwise'")
            
            if (i + 1) % 10 == 0 or (i + 1) == len(gen_results):
                print(f"Processed {i + 1}/{len(gen_results)} generated images, current similarity = {similarities[-1]:.4f}")

        if "treat_undetected_as_negative" in other_params and other_params["treat_undetected_as_negative"]: 
            print(f"Treating failed images as negative matches: {len(gen_failed)}")
            similarities += [0.0] * len(gen_failed)

        # Convert similarities to numpy array for easier computation
        similarities_array = np.array(similarities)
        
        # Compute final statistics
        stats = {
            'num_generated_images': len(generated_images),
            'num_ref_images': len(ref_images),
            'successful_generated': len(gen_results),
            'successful_ref': len(ref_results),
            'failed_generated': len(gen_failed),
            'failed_ref': len(ref_failed),
            'min_similarity': np.min(similarities_array),
            'max_similarity': np.max(similarities_array),
            'std_similarity': np.std(similarities_array),
            'median_similarity': np.median(similarities_array),
            'compute_option': compute_option,
            'batch_size': batch_size
        }
        
        if compute_option == 'pairwise':
            pairwise_similarities_array = np.array(pairwise_similarities)  # [num_generated, num_ref]
            stats.update({
                'pairwise_min': np.min(pairwise_similarities_array),
                'pairwise_max': np.max(pairwise_similarities_array),
                'pairwise_std': np.std(pairwise_similarities_array)
            })
        
        print(f"\nResults ({compute_option}):")
        print(f"Average similarity: {np.mean(similarities_array):.4f}")
        print(f"Min similarity: {stats['min_similarity']:.4f}")
        print(f"Max similarity: {stats['max_similarity']:.4f}")
        print(f"Std similarity: {stats['std_similarity']:.4f}")
        print(f"Successful generated: {stats['successful_generated']}/{stats['num_generated_images']}")
        print(f"Successful references: {stats['successful_ref']}/{stats['num_ref_images']}")
        
        if len(gen_failed) > 0:
            print(f"\nFailed generated images: {len(gen_failed)}")
            for path, error in list(gen_failed.items())[:5]:  # Show first 5 failures
                print(f"  {path}: {error}")
            if len(gen_failed) > 5:
                print(f"  ... and {len(gen_failed) - 5} more")
        
        if len(ref_failed) > 0:
            print(f"\nFailed reference images: {len(ref_failed)}")
            for path, error in list(ref_failed.items())[:5]:  # Show first 5 failures
                print(f"  {path}: {error}")
            if len(ref_failed) > 5:
                print(f"  ... and {len(ref_failed) - 5} more")
        
        if compute_option == 'pairwise':
            print(f"\nPairwise statistics:")
            print(f"All pairs min: {stats['pairwise_min']:.4f}")
            print(f"All pairs max: {stats['pairwise_max']:.4f}")
            print(f"All pairs std: {stats['pairwise_std']:.4f}")
        
        return np.mean(similarities_array), similarities, stats



class DinoFacePipeline:
    def __init__(self, arcface_app, device='cuda:0'):
        """
        Initialize the DINO face pipeline
        
        Args:
            arcface_app: Pre-initialized ArcFace application for face detection
        """
        self.arcface_app = arcface_app
        
        # Load DINO ViT-S/16 model
        # self.dino_model = ViTModel.from_pretrained('facebook/dino-vits16')
        
        self.dino_model = AutoModel.from_pretrained('facebook/dinov2-base')
        
        self.dino_model.to(device)
        self.dino_model.eval()
        
        # DINO transforms
        self.dino_transforms = transforms.Compose([
            transforms.Resize(256, interpolation=3),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
    
    def detect_and_crop_face(self, image_path, face_index=0):
        """
        Detect face in image and return cropped face
        
        Args:
            image_path (str): Path to input image
            face_index (int): Index of face to use if multiple faces detected
            
        Returns:
            PIL.Image: Cropped face image
            dict: Face detection info
        """
        # Read image using cv2 (BGR format) for face detection
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise ValueError(f"Could not read image: {image_path}")
        
        # Detect faces using BGR image
        faces = self.arcface_app.get(img_bgr)
        
        if len(faces) < 1:
            raise ValueError("No faces detected in the image")
        
        if len(faces) > 1:
            print(f"Warning: {len(faces)} faces detected. Using face at index {face_index}")
        
        if face_index >= len(faces):
            raise ValueError(f"Face index {face_index} out of range. Only {len(faces)} faces detected")
        
        # Get selected face bbox
        face = faces[face_index]
        bbox = face.bbox.astype(int)
        x1, y1, x2, y2 = bbox
        
        # Crop face from BGR image
        face_crop_bgr = img_bgr[y1:y2, x1:x2]
        
        # Convert BGR to RGB for PIL/DINO processing
        face_crop_rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
        face_pil = Image.fromarray(face_crop_rgb)
        
        face_info = {
            'bbox': bbox,
            'confidence': face.det_score,
            'face_index': face_index,
            'total_faces': len(faces),
            'face_crop_rgb': face_crop_rgb,
            'face_crop_pil': face_pil
        }
        
        return face_pil, face_info

    def batch_detect_and_crop_face(self, image_paths, face_index=0, batch_size=32):
        """
        Optimized batch processing for face detection and cropping.
        Since InsightFace doesn't have native batch processing, this implements
        optimizations like pre-loading images and reducing I/O overhead.
        
        Args:
            image_paths (list): List of image file paths
            face_index (int): Index of face to use if multiple faces detected
            batch_size (int): Number of images to process in each batch (for memory management)
            
        Returns:
            tuple: (successful_results, failed_results)
                - successful_results: Dict mapping image_path -> (face_pil, face_info)
                - failed_results: Dict mapping image_path -> error_message
        """
        successful_results = {}
        failed_results = {}
        
        print(f"Processing {len(image_paths)} images in batches of {batch_size} (optimized I/O)")
        
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]
            
            print(f"Processing batch {i//batch_size + 1}/{(len(image_paths) + batch_size - 1)//batch_size} "
                  f"({len(batch_paths)} images)")
            
            # Pre-load all images in the batch to reduce I/O overhead
            batch_data = []
            for img_path in batch_paths:
                try:
                    img_bgr = cv2.imread(img_path)
                    if img_bgr is not None:
                        batch_data.append((img_path, img_bgr))
                    else:
                        failed_results[img_path] = f"Could not read image: {img_path}"
                except Exception as e:
                    failed_results[img_path] = f"Error loading image: {str(e)}"
            
            if not batch_data:
                continue
            
            # Process each image in the pre-loaded batch
            # Note: InsightFace doesn't support true batch inference with app.get()
            # But we can optimize by reducing I/O and processing in chunks
            for img_path, img_bgr in batch_data:
                try:
                    # Use the standard InsightFace get() method
                    faces = self.arcface_app.get(img_bgr)
                    
                    if len(faces) < 1:
                        failed_results[img_path] = "No faces detected in the image"
                        continue
                    
                    if len(faces) > 1:
                        print(f"Warning: {len(faces)} faces detected in {img_path}. Using face at index {face_index}")
                    
                    if face_index >= len(faces):
                        failed_results[img_path] = f"Face index {face_index} out of range. Only {len(faces)} faces detected"
                        continue
                    
                    # Get selected face
                    face = faces[face_index]
                    bbox = face.bbox.astype(int)
                    x1, y1, x2, y2 = bbox
                    
                    # Crop face from BGR image
                    face_crop_bgr = img_bgr[y1:y2, x1:x2]
                    
                    # Convert BGR to RGB for PIL/DINO processing
                    face_crop_rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
                    face_pil = Image.fromarray(face_crop_rgb)
                    
                    face_info = {
                        'bbox': bbox,
                        'confidence': face.det_score,
                        'face_index': face_index,
                        'total_faces': len(faces),
                        'face_crop_rgb': face_crop_rgb,
                        'face_crop_pil': face_pil,
                        'image_path': img_path
                    }
                    
                    successful_results[img_path] = (face_pil, face_info)
                    
                except Exception as e:
                    failed_results[img_path] = f"Face processing error: {str(e)}"
            
            # Optional: Force garbage collection after each batch to manage memory
            import gc
            gc.collect()
        
        print(f"Batch processing complete: {len(successful_results)} successful, {len(failed_results)} failed")
        return successful_results, failed_results
    
    def get_dino_embedding(self, face_image):
        """
        Extract DINO embedding from face image
        
        Args:
            face_image (PIL.Image): Face image
            
        Returns:
            torch.Tensor: DINO embedding (CLS token)
        """
        # Apply DINO transforms
        face_tensor = self.dino_transforms(face_image).unsqueeze(0)  # Add batch dimension
        face_tensor = face_tensor.to(self.dino_model.device)  # Move to correct device
        # Get DINO features
        with torch.no_grad():
            outputs = self.dino_model(face_tensor)
        
        # Extract CLS token embedding
        last_hidden_states = outputs.last_hidden_state
        embedding = last_hidden_states[0, 0]  # First sample, CLS token
        
        return embedding
    
    def compute_face_similarity(self, image_path1, image_path2, face_index1=0, face_index2=0):
        """
        Compute similarity between faces in two images
        
        Args:
            image_path1 (str): Path to first image
            image_path2 (str): Path to second image
            face_index1 (int): Face index to use from first image
            face_index2 (int): Face index to use from second image
            
        Returns:
            dict: Results including similarity score and face info
        """
        # Detect and crop faces
        face1, face1_info = self.detect_and_crop_face(image_path1, face_index1)
        face2, face2_info = self.detect_and_crop_face(image_path2, face_index2)
        
        # Get DINO embeddings
        emb1 = self.get_dino_embedding(face1)
        emb2 = self.get_dino_embedding(face2)
        
        # Compute cosine similarity
        similarity = F.cosine_similarity(emb1, emb2, dim=0)
        
        results = {
            'similarity': similarity.item(),
            'face1_info': face1_info,
            'face2_info': face2_info,
            'face1_crop': face1,
            'face2_crop': face2,
            'embedding1': emb1,
            'embedding2': emb2
        }
        
        return results
    
    def get_image_files(self, directory):
        """
        Get all image files from a directory
        
        Args:
            directory (str): Path to directory
            
        Returns:
            list: List of image file paths
        """
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
        image_files = []
        
        for file_path in Path(directory).rglob('*'):
            if file_path.is_file() and file_path.suffix.lower() in image_extensions:
                image_files.append(str(file_path))
        
        return image_files
    
    def compute_dino_face_similarity(self, generated_path, ref_path, compute_option='pairwise', batch_size=32, other_params={}):
        """
        Compute DINO face similarity scores between generated and reference images using batch processing
        
        Args:
            generated_path (str): Directory containing generated images
            ref_path (str): Directory containing reference images
            compute_option (str): Method to compute similarity:
                'avg_ref' - compares against average reference embedding (default)
                'pairwise' - compares against each reference image individually
            batch_size (int): Batch size for processing images
        
        Returns:
            tuple: (average_similarity, individual_scores, stats)
                - average_similarity: Average cosine similarity score
                - individual_scores: List of similarity scores (method depends on compute_option)
                - stats: Dictionary containing additional statistics
        """
        # Get all image files from both directories
        generated_images = natsorted(self.get_image_files(generated_path))
        ref_images = self.get_image_files(ref_path)
        
        if not generated_images:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_images:
            raise ValueError(f"No images found in reference directory: {ref_path}")
        
        print(f"Found {len(generated_images)} generated images and {len(ref_images)} reference images")
        
        # Batch process reference images
        print("Processing reference images...")
        ref_results, ref_failed = self.batch_detect_and_crop_face(ref_images, batch_size=batch_size)
        
        if not ref_results:
            raise ValueError("No valid face embeddings extracted from reference images")
        
        # Extract DINO embeddings from reference images
        ref_embeddings = []
        for img_path, (face_pil, face_info) in ref_results.items():
            embedding = self.get_dino_embedding(face_pil)
            ref_embeddings.append(embedding)
        
        # Normalize all reference embeddings
        ref_embeddings = [F.normalize(emb, p=2, dim=0) for emb in ref_embeddings]
        ref_embeddings = torch.stack(ref_embeddings)
        
        # Batch process generated images
        print("Processing generated images...")
        gen_results, gen_failed = self.batch_detect_and_crop_face(generated_images, batch_size=batch_size)
        
        if not gen_results:
            raise ValueError("No valid similarities computed from generated images")
        
        # Compute similarities for each generated image
        similarities = []
        pairwise_similarities = []  # Only used for 'pairwise' mode
        
        print("Computing similarities...")
        for i, (img_path, (face_pil, face_info)) in enumerate(gen_results.items()):
            generated_embedding = self.get_dino_embedding(face_pil)
            generated_embedding = F.normalize(generated_embedding, p=2, dim=0)
            
            if compute_option == 'avg_ref':
                # Compare to average reference embedding
                avg_ref_embedding = torch.mean(ref_embeddings, dim=0)
                avg_ref_embedding = F.normalize(avg_ref_embedding, p=2, dim=0)
                similarity = F.cosine_similarity(generated_embedding, avg_ref_embedding, dim=0)
                similarities.append(similarity.item())
                
            elif compute_option == 'pairwise':
                # Compare to each reference individually
                pairwise_sims = F.cosine_similarity(generated_embedding.unsqueeze(0), 
                                                ref_embeddings)
                pairwise_similarities.append(pairwise_sims)
                similarities.append(torch.mean(pairwise_sims).item())  # Average of all pairwise similarities
                
            else:
                raise ValueError(f"Invalid compute_option: {compute_option}. Must be 'avg_ref' or 'pairwise'")
            
            if (i + 1) % 10 == 0 or (i + 1) == len(gen_results):
                print(f"Processed {i + 1}/{len(gen_results)} generated images, current similarity = {similarities[-1]:.4f}")
        
        if not similarities:
            raise ValueError("No valid similarities computed from generated images")
        
        
        if "treat_undetected_as_negative" in other_params and other_params["treat_undetected_as_negative"]: 
            print(f"Treating failed images as negative matches: {len(gen_failed)}")
            similarities += [0.0] * len(gen_failed)
            
        # Convert similarities to tensor for easier computation
        similarities_tensor = torch.tensor(similarities)
        
        # Compute final statistics
        stats = {
            'num_generated_images': len(generated_images),
            'num_ref_images': len(ref_images),
            'successful_generated': len(gen_results),
            'successful_ref': len(ref_results),
            'failed_generated': len(gen_failed),
            'failed_ref': len(ref_failed),
            'min_similarity': torch.min(similarities_tensor).item(),
            'max_similarity': torch.max(similarities_tensor).item(),
            'std_similarity': torch.std(similarities_tensor).item(),
            'median_similarity': torch.median(similarities_tensor).item(),
            'compute_option': compute_option,
            'batch_size': batch_size
        }
        
        if compute_option == 'pairwise':
            pairwise_similarities = torch.stack(pairwise_similarities)  # [num_generated, num_ref]
            stats.update({
                'pairwise_min': torch.min(pairwise_similarities).item(),
                'pairwise_max': torch.max(pairwise_similarities).item(),
                'pairwise_std': torch.std(pairwise_similarities).item()
            })
        
        print(f"\nResults ({compute_option}):")
        print(f"Average similarity: {torch.mean(similarities_tensor).item():.4f}")
        print(f"Min similarity: {stats['min_similarity']:.4f}")
        print(f"Max similarity: {stats['max_similarity']:.4f}")
        print(f"Std similarity: {stats['std_similarity']:.4f}")
        print(f"Successful generated: {stats['successful_generated']}/{stats['num_generated_images']}")
        print(f"Successful references: {stats['successful_ref']}/{stats['num_ref_images']}")
        
        if len(gen_failed) > 0:
            print(f"\nFailed generated images: {len(gen_failed)}")
            for path, error in list(gen_failed.items())[:5]:  # Show first 5 failures
                print(f"  {path}: {error}")
            if len(gen_failed) > 5:
                print(f"  ... and {len(gen_failed) - 5} more")
        
        if len(ref_failed) > 0:
            print(f"\nFailed reference images: {len(ref_failed)}")
            for path, error in list(ref_failed.items())[:5]:  # Show first 5 failures
                print(f"  {path}: {error}")
            if len(ref_failed) > 5:
                print(f"  ... and {len(ref_failed) - 5} more")
        
        if compute_option == 'pairwise':
            print(f"\nPairwise statistics:")
            print(f"All pairs min: {stats['pairwise_min']:.4f}")
            print(f"All pairs max: {stats['pairwise_max']:.4f}")
            print(f"All pairs std: {stats['pairwise_std']:.4f}")
        
        return torch.mean(similarities_tensor).item(), similarities, stats



In [13]:


arcface_app = FaceAnalysis(det_name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
# arcface_app = FaceAnalysis(det_name='buffalo_l')
arcface_app.prepare(ctx_id=0, det_thresh=0.05)


dinoface_pipeline = DinoFacePipeline(arcface_app)
arcface_pipeline = ArcFacePipeline(arcface_app)




Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/nessessence/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvid

# Morphing

In [ ]:
# morph_diffusion: computing arcface similarity scores

root_morph_folder = 'data_root/data/real_data/morph_ffhq/n30_v0'
other_params = {
    "treat_undetected_as_negative": True,  # Treat undetected faces as negative matches
}
          
summary_exp2scores = defaultdict(list)
for concept in tqmd(os.listdir(root_morph_folder)):
    concept_path = os.path.join(root_morph_folder, concept)
    print(f"num anchor images for concept {concept}: {len(os.listdir(concept_path))}")
    for id_ in os.listdir(concept_path):
        folder_path = os.path.join(concept_path, id_)
    
        gen_img_path = os.path.join(folder_path, 'morphed')
        ref_target_img_path = os.path.join(folder_path, 'img0')
        # ref_unseen_img_path = os.path.join(folder_path, 'unseen')

        labels = [f"{concept}_{id_}"]
        mean_similarity, similarities, stats_ = arcface_pipeline.compute_arcface_similarity(gen_img_path, ref_target_img_path, compute_option='pairwise', other_params=other_params)

        if data_info[concept]['unseen']:
            summary_exp2scores['unseen'].append(similarities)
        else:
            summary_exp2scores['seen'].append(similarities)

total_interpolation_steps = 20
image_indices = list(range(total_interpolation_steps))
plot_mean_confidence(summary_exp2scores,image_indices,title=f"seen vs unseen",ylabel='Arcface Similarty')

morph_lpips_ratio = [ i*100/total_interpolation_steps for i in list(range(total_interpolation_steps))]
plot_mean_confidence(summary_exp2scores,morph_lpips_ratio,title=f"seen vs unseen",ylabel='Arcface Similarty',xlabel='Morph Ratio (LPIPS)')


# torch.save(summary_exp2scores, "data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pt")


In [ ]:
# morph_diffusion: computing dino similarity scores 

root_morph_folder = 'data_root/data/real_data/morph_ffhq/n30_v0'
other_params = {
    "treat_undetected_as_negative": True,  # Treat undetected faces as negative matches
}
          

summary_exp2scores = defaultdict(list)
for concept in tqmd(os.listdir(root_morph_folder)):
    concept_path = os.path.join(root_morph_folder, concept)
    for id_ in os.listdir(concept_path):
        folder_path = os.path.join(concept_path, id_)
    
        gen_img_path = os.path.join(folder_path, 'morphed')
        ref_target_img_path = os.path.join(folder_path, 'img0')
        # ref_unseen_img_path = os.path.join(folder_path, 'unseen')

        labels = [f"{concept}_{id_}"]
        mean_similarity, similarities, stats_ = dinoface_pipeline.compute_dino_face_similarity(gen_img_path, ref_target_img_path, compute_option='pairwise', other_params=other_params)

        if data_info[concept]['unseen']:
            summary_exp2scores['unseen'].append(similarities)
        else:
            summary_exp2scores['seen'].append(similarities)

total_interpolation_steps = 20
image_indices = list(range(total_interpolation_steps))
plot_mean_confidence(summary_exp2scores,image_indices,title=f"seen vs unseen",ylabel='Dino Similarty')

morph_lpips_ratio = [ i*100/total_interpolation_steps for i in list(range(total_interpolation_steps))]
plot_mean_confidence(summary_exp2scores,morph_lpips_ratio,title=f"seen vs unseen",ylabel='Dino Similarty',xlabel='Morph Ratio (LPIPS)')


# torch.save(summary_exp2scores, "data_root/data/real_data/morph_ffhq/summary_exp2scores_dinov2.pt")


# Hypothesis Test

## Init

In [ ]:
def get_change_ratios(scores, morph_levels=[25], total_morph_steps=20, normalize_by_level=False):
    change_ratios = [] # change
    for morph_level in morph_levels:
        morph_idx = int(total_morph_steps * (morph_level / 100))
        for score in scores: 
            change_ratio = score[morph_idx] # ~ most align to our usecase
            # change_ratio = (score[0] - score[morph_idx]) / score[0]
            if normalize_by_level: # normalize by lpips change (morph_level)
                change_ratio /= (morph_level) # how much arcface similarity changes when lpips changes by 1 percentage point
            change_ratios.append(change_ratio)
    return change_ratios



def compute_means_and_ci(A, B=None, alpha=0.05):
    ## two-sided confidence intervals (alpha/2)
    """
    Compute means and confidence intervals for one or two groups.
    
    Args:
        A: Array of data for group A (required)
        B: Array of data for group B (optional, default=None)
        alpha: Significance level (default 0.05 for 95% CI)
    
    Returns:
        dict: Contains means, confidence intervals, margin of error, and other statistics
    """
    # Compute mean and CI for group A
    mean_A = np.mean(A)
    var_A = np.var(A, ddof=1)
    n_A = len(A)
    se_A = np.sqrt(var_A / n_A)
    
    # t-critical value for group A
    df_A = n_A - 1
    t_crit_A = stats.t.ppf(1 - alpha/2, df_A)
    margin_A = t_crit_A * se_A  # Margin of error
    ci_A = (mean_A - margin_A, mean_A + margin_A)
    
    result = {
        'mean_A': mean_A,
        'ci_A': ci_A,
        'margin_A': margin_A,  # Margin of error (t_crit * se)
        'se_A': se_A,
        'n_A': n_A,
        'df_A': df_A
    }
    
    # If group B is provided, compute additional statistics
    if B is not None:
        mean_B = np.mean(B)
        var_B = np.var(B, ddof=1)
        n_B = len(B)
        se_B = np.sqrt(var_B / n_B)
        
        # t-critical value for group B
        df_B = n_B - 1
        t_crit_B = stats.t.ppf(1 - alpha/2, df_B)
        margin_B = t_crit_B * se_B  # Margin of error
        ci_B = (mean_B - margin_B, mean_B + margin_B)
        
        # Difference statistics
        diff_mean = mean_A - mean_B
        se_diff = np.sqrt(var_A / n_A + var_B / n_B)
        
        # Degrees of freedom for difference (Welch-Satterthwaite)
        numerator = (var_A / n_A + var_B / n_B)**2
        denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                       (var_B / n_B)**2 / (n_B - 1))
        df_diff = numerator / denominator
        
        t_crit_diff = stats.t.ppf(1 - alpha/2, df_diff)
        margin_diff = t_crit_diff * se_diff  # Margin of error for difference
        ci_diff = (diff_mean - margin_diff, diff_mean + margin_diff)
        
        # Add group B and difference statistics to result
        result.update({
            'mean_B': mean_B,
            'ci_B': ci_B,
            'margin_B': margin_B,  # Margin of error for B
            'se_B': se_B,
            'n_B': n_B,
            'df_B': df_B,
            'mean_diff': diff_mean,
            'ci_diff': ci_diff,
            'margin_diff': margin_diff,  # Margin of error for difference
            'se_diff': se_diff,
            'df_diff': df_diff
        })
    
    return result


# Statistical Tests
def equivalence_test_toast(A, B, alpha=0.05, delta=0.05):
    # alpha = 0.05  # Significance level
    # # Step 1: Define equivalence threshold (Δ)
    # delta = 0.02 # Adjust based on your domain knowledge!
    
    A = np.array(A)
    B = np.array(B)

    # Step 2: Perform TOST (Two One-Sided Tests)
    # Test 1: Is (A - B) > -delta?
    t1, p1 = stats.ttest_ind(A, B - delta, alternative='greater')
    # Test 2: Is (A - B) < delta?
    t2, p2 = stats.ttest_ind(A, B + delta, alternative='less')
    
    # print(f'Seen greater than Unseen: t1 = {t1:.4f}, p1 = {p1:.4f}')
    # print(f'Seen less than Unseen: t2 = {t2:.4f}, p2 = {p2:.4f}')
    print(f'Test 1: Can we reject that Seen is worse than Unseen? p1 = {p1:.4f}')
    print(f'Test 2: Can we reject that Seen is better than Unseen? p2 = {p2:.4f}')

    # Overall p-value for equivalence
    p_equiv = max(p1, p2)  

    print(f"delta: {delta}")
    # Step 3: Decision
    if p_equiv < alpha:
        print(f"Reject null hypothesis: A and B are equivalent within the specified delta: {delta}.")
        print(f"Equivalence PROVEN (p = {p_equiv:.4f} < {alpha})")
        print("Conclusion: A and B are practically equivalent.")
        
        return True
    else:
        print(f"Equivalence NOT proven (p = {p_equiv:.4f} >= {alpha})")
        print("Conclusion: A and B might still differ meaningfully.")
        
        return False
    



def equivalence_test_toast(A, B, alpha=0.05, delta=0.05):
    # alpha = 0.05  # Significance level
    # # Step 1: Define equivalence threshold (Δ)
    # delta = 0.02 # Adjust based on your domain knowledge!
    
    A = np.array(A)
    B = np.array(B)

    # Step 2: Perform TOST (Two One-Sided Tests)
    # Test 1: Is (A - B) > -delta?
    t1, p1 = stats.ttest_ind(A, B - delta, alternative='greater')
    # Test 2: Is (A - B) < delta?
    t2, p2 = stats.ttest_ind(A, B + delta, alternative='less')
    
    print(f'Seen greater than Unseen: t1 = {t1:.4f}, p1 = {p1:.4f}')
    print(f'Seen less than Unseen: t2 = {t2:.4f}, p2 = {p2:.4f}')
    # Overall p-value for equivalence
    p_equiv = max(p1, p2)  

    print(f"delta: {delta}")
    # Step 3: Decision
    if p_equiv < alpha:
        print(f"Reject null hypothesis: A and B are equivalent within the specified delta: {delta}.")
        print(f"Equivalence PROVEN (p = {p_equiv:.4f} < {alpha})")
        print("Conclusion: A and B are practically equivalent.")
        
        return True
    else:
        print(f"Equivalence NOT proven (p = {p_equiv:.4f} >= {alpha})")
        print("Conclusion: A and B might still differ meaningfully.")
        
        return False
    

def find_min_delta(A,B):
    # Compute difference in means
    diff_mean = np.mean(A) - np.mean(B)

    # Standard error (Welch's t-test for unequal variances)
    var_A = np.var(A, ddof=1)  # Sample variance (ddof=1 for unbiased)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)

    # Degrees of freedom (Welch-Satterthwaite)
    df = (var_A / n_A + var_B / n_B)**2 / (
        (var_A / n_A)**2 / (n_A - 1) + (var_B / n_B)**2 / (n_B - 1)
    )

    # Critical t-value for 90% CI (α=0.05 for each tail in TOST)
    t_crit = stats.t.ppf(0.95, df)  # One-tailed (0.95 quantile)

    # 90% Confidence Interval
    ci_lower = diff_mean - t_crit * se
    ci_upper = diff_mean + t_crit * se

    # Minimal δ where equivalence holds
    delta_min = max(abs(ci_lower), abs(ci_upper))


    ## check
    assert equivalence_test_toast(A,B, alpha=0.05, delta=delta_min+0.001) # p-value = 0.05 (checked)
    assert not equivalence_test_toast(A,B, alpha=0.05, delta=delta_min-0.001)
    
    return delta_min



def find_superiority_margin(A, B, alpha=0.05):
    """
    Returns the minimal amount by which A statistically significantly exceeds B.
    
    Returns: 
        Positive value: A exceeds B by at least this amount (with 95% confidence)
        Zero or negative: A does not significantly exceed B
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # Critical t-value for one-sided test
    t_crit = stats.t.ppf(1 - alpha, df)
    
    # The minimal amount by which A exceeds B (lower bound of one-sided CI)
    superiority_margin = diff_mean - t_crit * se
    
    print(f"Mean of A: {np.mean(A):.4f}")
    print(f"Mean of B: {np.mean(B):.4f}")
    print(f"Raw difference (A - B): {diff_mean:.4f}")
    print(f"Minimal superiority margin: {superiority_margin:.4f}")
    
    if superiority_margin > 0:
        print(f"✓ A exceeds B by at least {superiority_margin:.4f} units (95% confidence)")
    else:
        print(f"✗ A does not significantly exceed B")
    
    return superiority_margin

    
    print(f"90% CI for μ_A - μ_B: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"Minimal δ for equivalence: {delta_min:.4f}")
    
    ## check
    assert equivalence_test_toast(A,B, alpha=0.05, delta=delta_min+0.001) # p-value = 0.05 (checked)
    assert not equivalence_test_toast(A,B, alpha=0.05, delta=delta_min-0.001)
    
    return delta_min



# effect size
def cohens_d(A, B):  
    n1, n2 = len(A), len(B)  
    s1, s2 = np.std(A, ddof=1), np.std(B, ddof=1)  # ddof=1 for sample SD  
    s_pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1 + n2 - 2))  
    return (np.mean(A) - np.mean(B)) / s_pooled  


def superior_test(A, B, alpha=0.05, delta=0):
    """
    Test if A is statistically significantly superior to B by at least delta.
    
    Args:
        A, B: Arrays of data
        alpha: Significance level
        delta: Minimum superiority margin to test
    
    Returns:
        True if A is superior to B by at least delta, False otherwise
    """
    A = np.array(A)
    B = np.array(B)
    
    # Perform one-sided t-test: H0: μ_A - μ_B <= delta vs H1: μ_A - μ_B > delta
    t_stat, p_value = stats.ttest_ind(A, B, alternative='greater', equal_var=False)
    
    # Adjust for delta: we're testing if (A - B) > delta
    # The t-test above tests if (A - B) > 0, so we need to adjust
    # by shifting the data
    diff_mean = np.mean(A) - np.mean(B)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Calculate the actual t-statistic for the superiority test
    t_stat_adj = (diff_mean - delta) / se
    
    # Degrees of freedom (Welch-Satterthwaite)
    df = (var_A / n_A + var_B / n_B)**2 / (
        (var_A / n_A)**2 / (n_A - 1) + (var_B / n_B)**2 / (n_B - 1)
    )
    
    # Calculate p-value for the superiority test
    p_value_adj = 1 - stats.t.cdf(t_stat_adj, df)
    
    print(f'Superiority test: A > B + {delta:.4f}')
    print(f't-statistic = {t_stat_adj:.4f}, p-value = {p_value_adj:.4f}')
    
    if p_value_adj < alpha:
        print(f"✓ A is superior to B by at least {delta:.4f} (p = {p_value_adj:.4f} < {alpha})")
        return True
    else:
        print(f"✗ A is NOT superior to B by at least {delta:.4f} (p = {p_value_adj:.4f} >= {alpha})")
        return False

def find_superiority_margin(A, B, alpha=0.05):
    """
    Returns the minimal amount by which A statistically significantly exceeds B.
    
    Returns: 
        Positive value: A exceeds B by at least this amount (with 95% confidence)
        Zero or negative: A does not significantly exceed B
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # Critical t-value for one-sided test
    t_crit = stats.t.ppf(1 - alpha, df)
    
    # The minimal amount by which A exceeds B (lower bound of one-sided CI)
    superiority_margin = diff_mean - t_crit * se
    
    print(f"Mean of A: {np.mean(A):.4f}")
    print(f"Mean of B: {np.mean(B):.4f}")
    print(f"Raw difference (A - B): {diff_mean:.4f}")
    print(f"Minimal superiority margin: {superiority_margin:.4f}")
    
    if superiority_margin > 0:
        print(f"✓ A exceeds B by at least {superiority_margin:.4f} units (95% confidence)")
    else:
        print(f"✗ A does not significantly exceed B")
    
    # Verification: Test that the margin is correct
    # At the margin, the p-value should be exactly alpha
    if superiority_margin > 0:
        # Test that at the margin + epsilon, we reject
        test_result_above = superior_test(A, B, alpha=alpha, delta=superiority_margin + 1e-6)
        # Test that at the margin - epsilon, we don't reject
        test_result_below = superior_test(A, B, alpha=alpha, delta=superiority_margin - 1e-6)
        
        print(f"Verification:")
        print(f"At margin + ε: {test_result_above} (should be True)")
        print(f"At margin - ε: {test_result_below} (should be False)")
    
    return superiority_margin




In [ ]:
def superior_test(A, B, delta=0, alpha=0.05):
    """
    Test if A is statistically superior to B by at least delta.
    
    H0: µ_A - µ_B ≤ delta
    H1: µ_A - µ_B > delta
    
    Returns:
        p_value: The p-value for the superiority test
        reject_H0: True if we reject H0 (A is superior to B by at least delta)
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # t-statistic
    t_stat = (diff_mean - delta) / se
    
    # One-sided p-value (right-tailed test)
    p_value = 1 - stats.t.cdf(t_stat, df)
    
    reject_H0 = p_value < alpha
    
    print(f"Superiority test: A > B + {delta}")
    print(f"Difference (A - B): {diff_mean:.4f}")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}")
    print(f"Conclusion: {'A is superior' if reject_H0 else 'No evidence of superiority'}")
    
    return p_value, reject_H0

def find_superiority_margin(A, B, alpha=0.05):
    """
    Returns the minimal amount by which A statistically significantly exceeds B.
    
    Returns: 
        Positive value: A exceeds B by at least this amount (with 95% confidence)
        Zero or negative: A does not significantly exceed B
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # Critical t-value for one-sided test
    t_crit = stats.t.ppf(1 - alpha, df)
    
    # The minimal amount by which A exceeds B (lower bound of one-sided CI)
    superiority_margin = diff_mean - t_crit * se
    
    print(f"Mean of A: {np.mean(A):.4f}")
    print(f"Mean of B: {np.mean(B):.4f}")
    print(f"Raw difference (A - B): {diff_mean:.4f}")
    print(f"Minimal superiority margin: {superiority_margin:.4f}")
    
    if superiority_margin > 0:
        print(f"✓ A exceeds B by at least {superiority_margin:.4f} units (95% confidence)")
    else:
        print(f"✗ A does not significantly exceed B")
    
    return superiority_margin

def noninferior_test(A, B, delta, alpha=0.05):
    """
    Test if A is non-inferior to B within margin delta.
    
    H0: µ_A - µ_B ≤ -delta  (A is inferior to B by more than delta)
    H1: µ_A - µ_B > -delta  (A is not inferior to B, or difference > -delta)
    
    Returns:
        p_value: The p-value for the non-inferiority test
        reject_H0: True if we reject H0 (A is non-inferior to B)
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # t-statistic - test if difference > -delta
    t_stat = (diff_mean - (-delta)) / se
    
    # One-sided p-value (right-tailed test)
    p_value = 1 - stats.t.cdf(t_stat, df)
    
    reject_H0 = p_value < alpha
    
    print(f"Non-inferiority test: A > B - {delta}")
    print(f"Difference (A - B): {diff_mean:.4f}")
    print(f"Non-inferiority margin: {delta}")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}")
    print(f"Conclusion: {'A is non-inferior' if reject_H0 else 'A may be inferior'}")
    
    return p_value, reject_H0

def find_noninferiority_margin(A, B, alpha=0.05):
    """
    Returns the maximal margin within which A is statistically non-inferior to B.
    
    Returns: 
        Positive value: A is non-inferior to B within this margin
        Negative value: A may be inferior to B
    """
    # Compute difference in means (A - B)
    diff_mean = np.mean(A) - np.mean(B)
    
    # Standard error (Welch's for unequal variances)
    var_A = np.var(A, ddof=1)
    var_B = np.var(B, ddof=1)
    n_A, n_B = len(A), len(B)
    se = np.sqrt(var_A / n_A + var_B / n_B)
    
    # Degrees of freedom (Welch-Satterthwaite)
    numerator = (var_A / n_A + var_B / n_B)**2
    denominator = ((var_A / n_A)**2 / (n_A - 1) + 
                   (var_B / n_B)**2 / (n_B - 1))
    df = numerator / denominator
    
    # Critical t-value for one-sided test
    t_crit = stats.t.ppf(1 - alpha, df)
    
    # The maximal non-inferiority margin (upper bound calculation)
    # Hack: this a bit weird..... 
    # We want the largest delta such that diff_mean > -delta - t_crit * se
    # Rearranged: delta > -diff_mean - t_crit * se
    noninferiority_margin = -diff_mean - t_crit * se
    
    print(f"Mean of A: {np.mean(A):.4f}")
    print(f"Mean of B: {np.mean(B):.4f}")
    print(f"Raw difference (A - B): {diff_mean:.4f}")
    print(f"Maximal non-inferiority margin: {noninferiority_margin:.4f}")
    
    if noninferiority_margin > 0:
        print(f"✓ A is non-inferior to B within margin of {noninferiority_margin:.4f} (95% confidence)")
    else:
        print(f"✗ A may be inferior to B")
    
    return noninferiority_margin

# Experiment

In [25]:
# summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pth", weights_only=False)
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pt")
morph_levels = list(range(5,51,5))

seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels)
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels)


margin = find_noninferiority_margin(seen_change_ratios, unseen_change_ratios)
print(f'max noninferiority margin: {margin}')


margin = find_superiority_margin(seen_change_ratios, unseen_change_ratios)
print(f'min superiority margin: {margin}')

 ## two-sided confidence intervals (alpha/2)
print(f"alpha: 0.10")
results = compute_means_and_ci(seen_change_ratios, unseen_change_ratios,alpha=0.10)
print(f"mean difference: {results['mean_diff']:.4f} +- {results['margin_diff']:.4f}")
print(f"upperbound: {results['ci_diff'][1]:.4f}")


print(f"alpha: 0.05")
results = compute_means_and_ci(seen_change_ratios, unseen_change_ratios,alpha=0.05)
print(f"mean difference: {results['mean_diff']:.4f} +- {results['margin_diff']:.4f}")
print(f"lowerbound, upperbound: {results['ci_diff'][0]:.4f} {results['ci_diff'][1]:.4f}")


/tmp/ipykernel_929790/108940603.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_a

Mean of A: 0.5249
Mean of B: 0.5338
Raw difference (A - B): -0.0088
Maximal non-inferiority margin: 0.0034
✓ A is non-inferior to B within margin of 0.0034 (95% confidence)
max noninferiority margin: 0.003416063197960926
Mean of A: 0.5249
Mean of B: 0.5338
Raw difference (A - B): -0.0088
Minimal superiority margin: -0.0143
✗ A does not significantly exceed B
min superiority margin: -0.014268038860449719
alpha: 0.10
mean difference: -0.0088 +- 0.0054
upperbound: -0.0034
alpha: 0.05
mean difference: -0.0088 +- 0.0065
lowerbound, upperbound: -0.0153 -0.0024


In [13]:
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pth", weights_only=False)
# summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pt")
morph_levels = list(range(5,51,5))
# morph_levels = list(range(5,96,5))
# morph_levels = list(range(5,11,5))
seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels)
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels)

# mean and sd
print("Seen Change Ratios - Mean:", np.mean(seen_change_ratios), "SD:", np.std(seen_change_ratios))
print("Unseen Change Ratios - Mean:", np.mean(unseen_change_ratios), "SD:", np.std(unseen_change_ratios))

delta = 0.02
equivalence_test_toast(seen_change_ratios, unseen_change_ratios, alpha=0.05, delta=delta)


delta_min = find_min_delta(seen_change_ratios, unseen_change_ratios)
print(f'delta min: {delta_min}')

delta_min = find_superiority_margin(seen_change_ratios, unseen_change_ratios)
print(f'sup delta min: {delta_min}')


effect_size = cohens_d(seen_change_ratios, unseen_change_ratios)
print(f'Effect Size (Cohen\'s d): {effect_size:.4f}')

Seen Change Ratios - Mean: 0.5300377 SD: 0.25002342
Unseen Change Ratios - Mean: 0.5359791 SD: 0.25716925
Seen greater than Unseen: t1 = 3.0359, p1 = 0.0012
Seen less than Unseen: t2 = -5.6018, p2 = 0.0000
delta: 0.02
Reject null hypothesis: A and B are equivalent within the specified delta: 0.02.
Equivalence PROVEN (p = 0.0012 < 0.05)
Conclusion: A and B are practically equivalent.
Seen greater than Unseen: t1 = 1.8609, p1 = 0.0314
Seen less than Unseen: t2 = -4.4269, p2 = 0.0000
delta: 0.014559066539934981
Reject null hypothesis: A and B are equivalent within the specified delta: 0.014559066539934981.
Equivalence PROVEN (p = 0.0314 < 0.05)
Conclusion: A and B are practically equivalent.
Seen greater than Unseen: t1 = 1.4290, p1 = 0.0765
Seen less than Unseen: t2 = -3.9950, p2 = 0.0000
delta: 0.01255906653993498
Equivalence NOT proven (p = 0.0765 >= 0.05)
Conclusion: A and B might still differ meaningfully.
delta min: 0.01355906653993498
Mean of A: 0.5300
Mean of B: 0.5360
Raw differe

In [23]:
superior_test(seen_change_ratios, unseen_change_ratios,delta=-0.01356)
# superior_test(unseen_change_ratios,seen_change_ratios,delta=0.013)

Superiority test: A > B + -0.0136
t-statistic = 1.6452, p-value = 0.0500
✓ A is superior to B by at least -0.0136 (p = 0.0500 < 0.05)


True

In [16]:
print(len(seen_change_ratios))

12000


In [ ]:
delta min: 0.01355906653993498


In [ ]:
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pth", weights_only=False)

morph_levels = list(range(5,51,5))
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels)
seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels)

# mean and sd
print("Seen Change Ratios - Mean:", np.mean(seen_change_ratios), "SD:", np.std(seen_change_ratios))
print("Unseen Change Ratios - Mean:", np.mean(unseen_change_ratios), "SD:", np.std(unseen_change_ratios))

delta = 0.03
equivalence_test_toast(seen_change_ratios, unseen_change_ratios, alpha=0.05, delta=delta)

In [ ]:
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_dinov2.pth", weights_only=False)

morph_levels = list(range(5,51,5))
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels)
seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels)

# mean and sd
print("Seen Change Ratios - Mean:", np.mean(seen_change_ratios), "SD:", np.std(seen_change_ratios))
print("Unseen Change Ratios - Mean:", np.mean(unseen_change_ratios), "SD:", np.std(unseen_change_ratios))

delta = 0.03
equivalence_test_toast(seen_change_ratios, unseen_change_ratios, alpha=0.05, delta=delta)

In [ ]:
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pth", weights_only=False)

morph_levels = list(range(5,51,5))
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels, normalize_by_level=True)
seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels, normalize_by_level=True)

# mean and sd
print("Seen Change Ratios - Mean:", np.mean(seen_change_ratios), "SD:", np.std(seen_change_ratios))
print("Unseen Change Ratios - Mean:", np.mean(unseen_change_ratios), "SD:", np.std(unseen_change_ratios))

delta = 0.001
equivalence_test_toast(seen_change_ratios, unseen_change_ratios, alpha=0.05, delta=delta)

In [ ]:
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_dinov2.pth", weights_only=False)

morph_levels = list(range(5,51,5))
unseen_change_ratios = get_change_ratios(summary_exp2scores['unseen'],morph_levels=morph_levels, normalize_by_level=True)
seen_change_ratios = get_change_ratios(summary_exp2scores['seen'],morph_levels=morph_levels, normalize_by_level=True)

# mean and sd
print("Seen Change Ratios - Mean:", np.mean(seen_change_ratios), "SD:", np.std(seen_change_ratios))
print("Unseen Change Ratios - Mean:", np.mean(unseen_change_ratios), "SD:", np.std(unseen_change_ratios))

delta = 0.001
equivalence_test_toast(seen_change_ratios, unseen_change_ratios, alpha=0.05, delta=delta)

In [ ]:

# MIXED Changes
summary_exp2scores = torch.load("data_root/data/real_data/morph_ffhq/summary_exp2scores_arcface.pth")

Force_normal = False
Force_Welch = False

total_interpolation_steps = 20

# early_percents = [5, 10, 15, 20, 25]
early_percents = [5, 10, 15, 20, 25, 30, 35,40,45, 50]
early_percents =  [5, 10, 15, 20, 25,]
early_percents =  [5, 10]
# early_percents =  [50]
# early_percents = range(5, 51, 5)  # From 5% to 100% in steps of 5%
# early_percents = range(5, 96, 5)  # From 5% to 100% in steps of 5%
# early_percent = 20
seen_changes = []
unseen_changes = []



for early_percent in early_percents:
    print(f"Early percent: {early_percent}%")
        
        # Calculate the step corresponding to the early percent
    early_step = int(total_interpolation_steps * early_percent / 100)

    for seen_similarities in summary_exp2scores['seen']:
        seen_changes.append((seen_similarities[0]-seen_similarities[early_step])/seen_similarities[0])
        # seen_changes.append((seen_similarities[0]-seen_similarities[early_step])/early_percent)
        # seen_changes.append((seen_similarities[0]-seen_similarities[early_step])/(seen_similarities[0]*early_percent*0.01)) # equivalent of normlized arcface similarity change (slope)
        # seen_changes.append((seen_similarities[0]-seen_similarities[early_step])/((seen_similarities[0]-seen_similarities[-1])*early_percent*0.01)) # equivalent of normlized arcface similarity change (slope) but also set the lowest to zero
    for unseen_similarities in summary_exp2scores['unseen']:
        unseen_changes.append((unseen_similarities[0]-unseen_similarities[early_step])/unseen_similarities[0])
        # unseen_changes.append((unseen_similarities[0]-unseen_similarities[early_step])/early_percent)
        # unseen_changes.append((unseen_similarities[0]-unseen_similarities[early_step])/(unseen_similarities[0]*early_percent*0.01))
        # unseen_changes.append((unseen_similarities[0]-unseen_similarities[early_step])/((unseen_similarities[0]-unseen_similarities[-1])*early_percent*0.01))
    # print(f"seen changes: {seen_changes}")
    # print(f"unseen changes: {unseen_changes}")

print(len(seen_changes), len(unseen_changes))

A = np.array(seen_changes)
B = np.array(unseen_changes)

# stats
print('Mean seen change:', np.mean(A))
print('Mean unseen change:', np.mean(B))